# Chat logic

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(base_url="http://127.0.0.1:1234/v1", model="gemma-3-27b-it", api_key="random")

In [2]:
from langchain.schema import HumanMessage, AIMessage

def handle_message(message, history):
    history_langchain_format = []
    for msg in history:
        if msg['role'] == "user":
            history_langchain_format.append(HumanMessage(content=msg['content']))
        elif msg['role'] == "assistant":
            history_langchain_format.append(AIMessage(content=msg['content']))
    history_langchain_format.append(HumanMessage(content=message))
    llm_response = model.invoke(history_langchain_format)
    return llm_response.content

In [3]:
# Schema for structured output
from pydantic import BaseModel, Field

class HotelSearchRequirements(BaseModel):
    destination: str = Field(default=None, description="The destination where to look for hotels")
    num_adults: int = Field(None, description="Number of Adults in the trip")
    num_children: int = Field(None, description="Number of children in the strip")
    from_date: str = Field(None, description="Start date of trip")
    to_date: str = Field(None, description="End date of trip. This is the date till which they want to book the hotel.")
    num_rooms: int = Field(None, description="Number of rooms")
    amenities: list = Field(None, description="Amenities required")
    free_breakfast: bool = Field(None, description="Is free breakfast required?")
    free_cancelation: bool = Field(None, description="Is free cancelation required?")
    max_price: float = Field(None, description="What is the max budget?")
    min_price: float = Field(None, description="Any min cost?")

model_with_output = model.with_structured_output(HotelSearchRequirements)

In [4]:
obj = HotelSearchRequirements()

for key in obj:
    print(f"key: {key}")
    print(f"key[0]: {key[0]}")
    try:
        print(f"value: {obj.model_dump()[key[0]]}")
    except Exception as err:
        print("Didn't work!")
        print(err)

key: ('destination', None)
key[0]: destination
value: None
key: ('num_adults', None)
key[0]: num_adults
value: None
key: ('num_children', None)
key[0]: num_children
value: None
key: ('from_date', None)
key[0]: from_date
value: None
key: ('to_date', None)
key[0]: to_date
value: None
key: ('num_rooms', None)
key[0]: num_rooms
value: None
key: ('amenities', None)
key[0]: amenities
value: None
key: ('free_breakfast', None)
key[0]: free_breakfast
value: None
key: ('free_cancelation', None)
key[0]: free_cancelation
value: None
key: ('max_price', None)
key[0]: max_price
value: None
key: ('min_price', None)
key[0]: min_price
value: None


In [4]:
from typing import Annotated

from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]
    users: dict[str, HotelSearchRequirements]
    input: str
    destination: str
    num_adults: int
    num_children: int
    from_date: str
    to_date: str
    num_rooms: int
    amenities: list
    bed_types: list
    free_breakfast: bool
    free_cancelation: bool
    max_price: float
    min_price: float
    search_results: str
    extracted: bool
    who: str

def render_state_as_info(state: State):
    curr_user_req = state["users"][state["who"]]
    string_to_return = ""
    is_ready_for_tool_call = True
    for user in state["users"]:
        user_req = state["users"][user].model_dump()
        amenities_as_string: str = ",".join(user_req.get("amenities", [])) if user_req.get("amenities", []) is not None else None
        free_breakfast = "yes" if user_req.get("free_breakfast", None) is True else "no"
        free_cancelation = "yes" if user_req.get("free_cancelation", None) is True else "no"
        missing_fields_for_user = ""
        is_ready_for_tool_call_for_user = True
        users_from_whom_information_is_still_required = ""
        for key in HotelSearchRequirements.model_fields:
            if state.get(key, None) is None:
                if is_ready_for_tool_call_for_user:
                    missing_fields_for_user += key
                else:
                    missing_fields_for_user += f", {key}"
                is_ready_for_tool_call_for_user = False
        if not is_ready_for_tool_call_for_user:
            if is_ready_for_tool_call_for_user:
                users_from_whom_information_is_still_required += user
            else:
                users_from_whom_information_is_still_required += f", {user}"
            is_ready_for_tool_call = False
        string_to_return += f"""
            For user: {user}

            Destination: {user_req.get("destination", None)}
            Number of adults: {user_req.get("num_adults", None)}
            Number of children: {user_req.get("num_children", None)}
            Start date: {user_req.get("from_date", None)}
            End date: {user_req.get("to_date", None)}
            Number of rooms: {user_req.get("num_rooms", None)}
            Required amenities: {amenities_as_string}
            Is free breakfast required: {free_breakfast}
            Is free cancelation required: {free_cancelation}
            Max price: {user_req.get("max_price", None)}
            Min price: {user_req.get("min_price", None)}
            Extracted: {state["extracted"]}

            Conclusion about information you have from {user}:
            This means that you are{"" if is_ready_for_tool_call_for_user else " not"} ready to call search_hotels tool.
            {"" if is_ready_for_tool_call_for_user else f"The following fields are still missing: {missing_fields_for_user}"} 
            {"" if is_ready_for_tool_call_for_user else f"IMPORTANT!!! DO NOT CALL search_hotels TOOL."}
        """

    string_to_return += f"""
        Conclusion about information you have so far and whether you can call the search hotel tool:
    """
    if not is_ready_for_tool_call:
        string_to_return += f"""
            YOU MUST NOT CALL THE SEARCH HOTEL TOOL YET. 
            THERE'S MISSING INFORMATION FROM THE FOLLOWING USERS: {users_from_whom_information_is_still_required}
            YOU MUST OBTAIN ALL MISSING INFORMATION BEFORE YOU CAN PROCEED TO CALL THE SEARCH HOTEL TOOL
        """
    else:
        amenities_as_string: str = ",".join(state.get("amenities", [])) if state.get("amenities", []) is not None else None
        free_breakfast = "yes" if state.get("free_breakfast", None) is True else "no"
        free_cancelation = "yes" if state.get("free_cancelation", None) is True else "no"
        string_to_return += f"""
            YOU ARE READY TO CALL THE SEARCH HOTEL TOOL WITH THE FOLLOWING INFORMATION:

            Destination: {state.get("destination", None)}
            Number of adults: {state.get("num_adults", None)}
            Number of children: {state.get("num_children", None)}
            Start date: {state.get("from_date", None)}
            End date: {state.get("to_date", None)}
            Number of rooms: {state.get("num_rooms", None)}
            Required amenities: {amenities_as_string}
            Is free breakfast required: {free_breakfast}
            Is free cancelation required: {free_cancelation}
            Max price: {state.get("max_price", None)}
            Min price: {state.get("min_price", None)}
        """
    return string_to_return

graph_builder = StateGraph(State)

In [6]:
# Schema for structured output
from pydantic import BaseModel, Field
import enum

class HotelAmenityDetail(BaseModel):
    icon: str = Field(None, description="Icon that visually represents an amenity.")
    name: str = Field(None, description="Name of the amenity.")
    tags: list[str] = Field(None, description="List of tags associated with the amenity.")

class HotelType(enum.Enum):
    STAY_NOW = 'STAY_NOW'
    ASI = 'ASI'
    SABRE = 'SABRE'
    SITE_MINDER = 'SITE_MINDER'
    CLOUDBEDS = 'CLOUDBEDS'

class HotelDetails(BaseModel):
    address: str = Field(None, description="Address of the hotel.")
    amenities: list[HotelAmenityDetail] = Field(None, description="List of amenities that the hotel provides.")
    code: str = Field(None, description="Unique code representing the hotel.")
    distance: float = Field(None, description="Distance from destination.")
    guestRating: float = Field(None, description="Guest rating of hotel.")
    hotelGroup: str = Field(None, description="Hotel Group that owns the hotel.")
    hotelType: HotelType
    intro: str = Field(None, description="An introduction about the hotel.")
    logo: str  = Field(None, description="Hotel's logo.")
    name: str = Field(None, description="The name of the hotel.")
    reviewsCount: int = Field(None, description="Count of how many people reviewed the hotel.")
    summary: str = Field(None, description="A summary describing the hotel.")


class HotelSearchResults(BaseModel):
    hotels: list[str] = Field(None, description="List of hotels that met the search requirements.")


    def render_as_string(self):
        string = ""
        for hotel in self.hotels:
            string += f"""
                Hotel Name: {hotel["name"]}
                Summary: {hotel["summary"]}
            """

        if len(string) < 5:
            string = "No results"
        print(f"string being returned: {string}")

        return string

class HotelSearchRequestFilters(BaseModel):
    amenities: list[str] = Field(None, description="List of amenities that the hotel must have.")
    bedTypes: list[str] = Field(None, description="List of bed types that the hotel must have.")
    freeBreakfast: bool = Field(None, description="Should the hotel provide free breakfast?")
    freeCancellation: bool = Field(None, description="Should the hotel provide free cancellation?")
    maxPrice: float = Field(None, description="Upper boundary for hotel price.")
    minPrice: float = Field(None, description="Lower boundary for hotel price.")

class HotelSearchRequest(BaseModel):
    adults: int = Field(None, description="Number of adults to book for.")
    children: int = Field(None, description="Number of children to book for.")
    filters: HotelSearchRequestFilters = Field(None, description="Filters to apply when searching for hotel.")
    rooms: int = Field(None, description="Number of rooms to book.")
    fromDate: str = Field(None, description="The date from which the booking should be made.")
    toDate: str = Field(None, description="The date to which the booking should be made.")
    type: str = Field(None, description="Type of search. Will always be set to 'LOCATION'.")

def request_translator(requirements: HotelSearchRequirements) -> HotelSearchRequest:
    return HotelSearchRequest(
        adults=requirements.num_adults, children=requirements.num_children, rooms=requirements.num_rooms,
        fromDate=requirements.from_date, toDate=requirements.to_date, type="LOCATION",
        filters=HotelSearchRequestFilters(
            amenities=requirements.amenities, bedTypes=[],
            freeBreakfast=requirements.free_breakfast, freeCancellation=requirements.free_cancelation,
            maxPrice=requirements.max_price, minPrice=requirements.min_price
        )
    )

In [7]:
from gql import gql, Client
from gql.transport.aiohttp import AIOHTTPTransport
from langchain_core.tools import tool
from langgraph.types import Command
import dateutil.parser as parser

# Select your transport with a defined url endpoint
transport = AIOHTTPTransport(url="https://staging-api.inn3.com/graphql", headers={
   'Authorization': 'eyJraWQiOiJkLTE3NDI5MTMzNDEzMDQiLCJ0eXAiOiJKV1QiLCJ2ZXJzaW9uIjoiNCIsImFsZyI6IlJTMjU2In0.eyJpYXQiOjE3NDMyMzkwNTEsImV4cCI6MTc0MzI0MjY1MSwic3ViIjoiZDM2OWFjNWEtZThhZS00NzVmLWIwMGItOWRlMDUzODJmMGM3IiwidElkIjoicHVibGljIiwic2Vzc2lvbkhhbmRsZSI6IjczYjMzYWZiLTkzYjctNGQzMC05Zjc2LTk2MGZkZTM0NzhiMSIsInJlZnJlc2hUb2tlbkhhc2gxIjoiM2QyODlhY2Q2MjJmNzQwYWNlZDA1OWU0OTU0NDYxNjExYTRkZTUyMjA1ZjlkYjMxODQwZjM4ZjkyZWJjM2ZjMiIsInBhcmVudFJlZnJlc2hUb2tlbkhhc2gxIjpudWxsLCJhbnRpQ3NyZlRva2VuIjpudWxsLCJtZXRhZGF0YSI6eyJpZCI6IjY3ZTUzYTYyMTI4MjQ0NzEwM2RiNjljYSJ9LCJzdGF0dXMiOiJPSyIsImlzcyI6Imh0dHA6Ly9sb2NhbGhvc3QvYXV0aCIsInN0LWV2Ijp7InYiOnRydWUsInQiOjE3NDMyMzkwNTEyNzh9fQ.OD5FYplChpmn1I_Kwp0NHFi2whJ8iTFOxASfQwHs6O0ReUlPOQng3azrVvMkiD_9GCT1OgOOlbtkV4y5fc_PWX0jlVuv_mS5udc6kbD9_fPGt0yA3ocKx-UWoRntDLOJSyh6rOJnLUDgqzCm6szulWURqL75B7FbwW9huntpYmhxKBCwFuVVTxT90ZGa6j9PjQ9Cmt9JqxJ0ZpblGpcdR7WN0gb45Q4UzKUK9afWaiXj11m_oeJHoi4mDNOMU9lCarYL6yHxGqzbMef1nmDT-7P4RLVnXa-uW8T6WjlY7sl0Y6lRtL74m4cXmIEiqwqYVDO0ENp3VTo8ncgJ831lTA'
})

# Create a GraphQL client using the defined transport
client = Client(transport=transport, fetch_schema_from_transport=True)

async def search_location(requirements: HotelSearchRequirements) -> str:
    """Search for location

    Args:
      requirements: hotel requirements to be used for searching.
    """
    print(f"Search location tool was called with the requirements: {requirements}")

    gql_query = gql(
        """
        query SearchGlobally($input: GlobalSearchInput!) {
          searchGlobally(input: $input) {
            id
            name
            city
            state
            image
            type
            serviceable
          }
        }
        """
    )

    # Execute the query on the transport
    response = await client.execute_async(gql_query, variable_values={"input": {
        "keyword": requirements.destination
    }})

    print("response: ")
    print(response)

    for result in response['searchGlobally']:
      if result["name"].lower() == requirements.destination.lower() and result["type"] == "LOCATION":   
         print("id obtained!")
         print(result["id"])
         return result["id"]     

@tool
async def search_hotels(query: HotelSearchRequirements) -> HotelSearchResults:
    """Search for hotels

    Args:
        query: hotel requirements to be used for searching
    """
    print(f"Search tool was called with the query: {query}")

    # location_id = await search_location(requirements=query)
    location_id = "343"

    request = request_translator(query)

    print(f"The request being sent is {request}")

    # Provide a GraphQL query
    gql_query = gql(
        """
        query SearchAvailability($input: SearchAvailabilityCombinedInput!) {
              searchAvailability(input: $input) {
                hotels {
                  id
                  roomInventoryIds {
                    ids
                    roomType
                  }
                  hotelGroup
                  name
                  intro
                  summary
                  logo
                  code
                  starRating
                  guestRating
                  reviewsCount
                  type
                  address {
                    line1
                    line2
                    locality {
                      id
                      name
                    }
                    city {
                      id
                      name
                    }
                    county {
                      id
                      name
                    }
                    state {
                      id
                      name
                    }
                    country {
                      id
                      name
                    }
                    zipcode
                    timezone
                    currency
                  }
                  media {
                    mediaId
                    type
                    caption {
                      title
                      subtitle
                    }
                    visibility
                    isVertical
                    thumbnail
                    categories
                    category
                    name
                  }
                  isWishlisted
                  wishlistedTripIds
                  roomTypes {
                    roomTypeId
                    ratePlans {
                      ratePlanId
                      totalCharge
                      totalChargeWithTax
                    }
                    roomTypeName
                  }
                  minTotalCharge {
                    ratePlanId
                    totalCharge
                    totalChargeWithTax
                  }
                  distance
                  isWishtlisted
                  slug
                  hotelType
                  amenities {
                    name
                    icon
                    tags
                  }
                }
                filterOptions {
                  priceRange {
                    minPrice
                    maxPrice
                  }
                  popularFilters {
                    label
                    type
                    value
                    count
                    icon
                  }
                  guestRatings {
                    label
                    type
                    value
                    count
                    icon
                  }
                  starRatings {
                    label
                    type
                    value
                    count
                    icon
                  }
                  bedTypes {
                    label
                    type
                    value
                    count
                    icon
                  }
                  amenities {
                    label
                    type
                    value
                    icon
                  }
                }
                location {
                  name
                  latitude
                  longitude
                }
                destination {
                  id
                  name
                  description
                  address {
                    city
                    state
                    country
                  }
                  aarRadius
                  aarRefreshCycle
                  isShownToGuests
                  latitude
                  longitude
                  isActive
                  categories
                  status
                  weatherUnit
                  hotels
                  aars
                  relatedZipcodes
                  zoomLevel
                  banner
                  slug
                  boundary {
                    type
                    coordinates
                  }
                  createdAt
                  updatedAt
                  locationDetails {
                    city
                    state
                  }
                }
                aar {
                  id
                  refId
                  googlePlaceId
                  type
                  name
                  intro
                  summary
                  location {
                    type
                    coordinates
                  }
                  tags
                  platformRank
                  rank
                  rating
                  reviewsCount
                  link
                  isApproved
                  isActive
                  isFreeOfCost
                  isParkingAvailable
                  aarTimings {
                    monday {
                      startTime
                      endTime
                      isOpen24Hrs
                      isClosed
                    }
                    tuesday {
                      startTime
                      endTime
                      isOpen24Hrs
                      isClosed
                    }
                    wednesday {
                      startTime
                      endTime
                      isOpen24Hrs
                      isClosed
                    }
                    thursday {
                      startTime
                      endTime
                      isOpen24Hrs
                      isClosed
                    }
                    friday {
                      startTime
                      endTime
                      isOpen24Hrs
                      isClosed
                    }
                    saturday {
                      startTime
                      endTime
                      isOpen24Hrs
                      isClosed
                    }
                    sunday {
                      startTime
                      endTime
                      isOpen24Hrs
                      isClosed
                    }
                  }
                  photoOfHours
                  attractionTypes
                  phone
                  email
                  website
                  blogs {
                    link
                    title
                    thumbnailUrl
                    source
                  }
                  address {
                    line1
                    line2
                    state
                    country
                    zipcode
                  }
                  typeOfCuisine
                  priceCategory
                  linkToMenu
                  isParkingPaid
                  isPricingPaid
                  linkToReserveTable
                  status
                  pricingStartsFrom
                  linkToPurchaseTickets
                  specialDiets
                  highlights
                  activityTypes
                  currency
                  destinations {
                    id
                    zoom
                  }
                  zoomLevel
                  createdAt
                  updatedAt
                  isWishlisted
                }
              }
            }
        """
    )

    # Execute the query on the transport
    var_values = request.model_dump()
    var_values["id"] = location_id
    var_values['fromDate'] = parser.parse(request.fromDate).isoformat()
    var_values['toDate'] = parser.parse(request.toDate).isoformat()

    print(var_values)
    result = await client.execute_async(gql_query, variable_values={"input": var_values})

    print(result)

    hotels = HotelSearchResults(hotels=result["searchAvailability"]["hotels"])

    return Command(
            update={
                # update the state keys
                "search_results": hotels.render_as_string()
            }
        )

tools = [search_hotels]

model_with_tools = model.bind_tools(tools)

/Users/joat/Documents/Projects/multiuserchat2/env/lib/python3.12/site-packages/gql/transport/aiohttp.py:92: UserWarning: WARNING: By default, AIOHTTPTransport does not verify ssl certificates. This will be fixed in the next major version. You can set ssl=True to force the ssl certificate verification or ssl=False to disable this warning
  warnings.warn(


In [8]:
# await search_location(HotelSearchRequirements(destination="Amsterdam", num_adults=5, num_rooms=3, num_children=2, free_breakfast=True, free_cancelation=True, max_price=2000, min_price=1000, amenities=["pool"], from_date="20-05-2025", to_date="25-05-2025"))

In [9]:
print(tools)

[StructuredTool(name='search_hotels', description='Search for hotels\n\n    Args:\n        query: hotel requirements to be used for searching', args_schema=<class 'langchain_core.utils.pydantic.search_hotels'>, coroutine=<function search_hotels at 0x117468cc0>)]


In [10]:
from langchain_core.messages import ToolMessage
import json


class BasicToolNode:
    """A node that runs the tools requested in the last AIMessage."""

    def __init__(self, tools: list) -> None:
        self.tools_by_name = {tool.name: tool for tool in tools}

    async def __call__(self, inputs: dict):
        if messages := inputs.get("messages", []):
            message = messages[-1]
        else:
            raise ValueError("No message found in input")
        for tool_call in message.tool_calls:
            output = await self.tools_by_name[tool_call["name"]].ainvoke(
                tool_call["args"]
            )
            return output

tool_node = BasicToolNode(tools=tools)

In [11]:
from langgraph.graph import END

def should_extract_or_end(state: State):
    for key in HotelSearchRequirements.model_fields:
        if state.get(key, None) is None:
            if state["extracted"]:
                print(f"Ending because {key} is unset and extracted is true")
                print(f"value of extracted: {state["extracted"]}")
                return END
            return "chatbot"
    return False

def route_tools(
    state: State,
):
    """
    Use in the conditional_edge to route to the ToolNode if the last message
    has tool calls. Otherwise, route to the end.
    """
    print("route tools was called!!")
    if isinstance(state, list):
        ai_message = state[-1]
    elif messages := state.get("messages", []):
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    should_extract_or_end_value = should_extract_or_end(state)
    if should_extract_or_end_value is not False:
        return should_extract_or_end_value
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        for tool_call in ai_message.tool_calls:
            print("tool_call obj: ")
            print(tool_call)
            try:
                print("tool_call.function")
                print(tool_call.function)
            except:
                print("tool_call.function not working")
            try:
                print("tool_call['function']")
                print(tool_call['function'])
            except:
                print("tool_call['function'] not working")
            if tool_call['name'] == "search_hotels":
                return "tools"
    return END

In [12]:
from langchain_core.messages import HumanMessage, SystemMessage

async def get_group_info(state: State):

    return {
        "messages": [await model.ainvoke([
            SystemMessage(content=f"""
                You are the world's best travel agent who has information about all the hotels in the world.
                
                Today your goal is to help a group of users choose the right hotel for their needs.
                The entire group wants to travel together and stay at the same hotel.
                          
                But you don't know who 
            """)
        ])]
    }

async def chatbot(state: State):

    print(f"rendered state: {render_state_as_info(state)}")
    return {"messages": [await model_with_tools.ainvoke([
            SystemMessage(content=f"""
                You are the world's best travel agent who has information about all the hotels in the world.
                
                Today your goal is to help a group of users choose the right hotel for their needs.
                The entire group wants to travel together and stay at the same hotel.
                
                You can can find a hotel according to the needs of anyone. You do so by using the search hotel tool.
                However, you need certain information from the users to do so.
                This includes destination, number of adults, number of children, number of rooms, start date of trip, end date of trip, amenities required, required bed types, whether or not free breakfast should be included (free_breakfast), what the budget range is as well (max_price and min_price).
                
                You need to obtain this information with the smallest amount of questions and as quick as possible without appearing like a robot or a form.
                You can search for a hotel using the search_hotels tool once you have all the required information.

                You can only invoke the search tool if the text below says you're ready to call. Otherwise you must ask the user for the information that's missing below
                Here's the information you already have (Note: A value of 'None' means you don't yet have the information and you need to ask the user for it.): 
                {render_state_as_info(state)}

                Here's the latest input from: 
            """),
            HumanMessage(content=state["input"]),
        ])
    ]}

async def extract_and_save(state: State):
    print("extract_and_Save called!")
    output: HotelSearchRequirements | None = await model_with_output.ainvoke([
        SystemMessage(content=f"Extract whatever information is available from the user input"),
        HumanMessage(content=state["input"])
    ])
    print("output: ")
    print(output)
    output_as_json = {}
    extracted_data = {"users": {"joseph": state["users"]["joseph"], "saawan": state["users"]["saawan"]}}
    if output is not None:
        output_as_json = output.model_dump()
    print("output_as_json: ")
    print(output_as_json)
    curr_user_req = state["users"][state["who"]].model_dump()
    for key in output_as_json:
        if key in curr_user_req and curr_user_req[key] is not None and output_as_json[key] is None:
            output_as_json[key] = curr_user_req[key]
    converted_output = HotelSearchRequirements(**output_as_json)
    extracted_data["users"][state["who"]] = converted_output
    print("setting extracted to true")
    extracted_data["extracted"] = True
    print("updated extracted_data: ")
    print(extracted_data)
    return extracted_data

async def combine_requirements(state: State):
    print("combine_requirements called!")
    input_str = ""
    for user in state["users"]:
        user_req = state["users"][user].model_dump()
        amenities_as_string: str = ",".join(user_req.get("amenities", [])) if user_req.get("amenities", []) is not None else None
        free_breakfast = "yes" if user_req.get("free_breakfast", None) is True else "no"
        free_cancelation = "yes" if user_req.get("free_cancelation", None) is True else "no"
        input_str += f"""
            For user: {user}

            Destination: {user_req.get("destination", None)}
            Number of adults: {user_req.get("num_adults", None)}
            Number of children: {user_req.get("num_children", None)}
            Start date: {user_req.get("from_date", None)}
            End date: {user_req.get("to_date", None)}
            Number of rooms: {user_req.get("num_rooms", None)}
            Required amenities: {amenities_as_string}
            Is free breakfast required: {free_breakfast}
            Is free cancelation required: {free_cancelation}
            Max price: {user_req.get("max_price", None)}
            Min price: {user_req.get("min_price", None)}
        """
    output = await model_with_output.ainvoke([
        SystemMessage(content=f"""
                    You are a travel agent responsible for understanding the requirements of a group of people.
                    As part of your job you have requirements from different people in a group and you have to find a common ground between them.
                    This means you have to take different values for the same requirements and come up with values for those requirements that represents the group in the best way possible.

                    You can follow the following rules to make the process simpler:
                    1. For each requirement, if you observe the same value multiple times, you should take the most common value
                    2. For the same requirement, if you find different values from every member of the group, you must follow the following sub rules:
                      2.1 If the requirement represents a range, take the smallest start value as the start value of the new range and the largest end value as the end of value of the new range. This is useful for things like budget range and date range.
                      2.2 If the requirement represents a numerical value other than for a range, take the smallest value.
                      2.3 If the requirement represents discrete values, take the union of the values.

                    !!!Note: If you don't have value for a requirement, you should populate it with the pythonic None value as the values are being used in python.
                    !!!IMPORTANT: null is NOT an acceptable value. DONOT populate with null. YOU MUST REPLACE NULL WITH PYTHONIC None.
                    
                    The requirements from each user is given below as input:
        """
        ),
        HumanMessage(content=input_str)
    ])

    print("Summarised version: ")
    print(output)
    extracted_data = output.model_dump()

    for key in extracted_data:
        if key in state and state[key] is not None and extracted_data[key] is None:
            extracted_data[key] = state[key]
    return extracted_data

async def process_results(state: State):
    return {
        "messages": [await model.ainvoke([
            SystemMessage(content=f"""
                You are the world's best travel agent who has information about all the hotels in the world.
                
                You have already searched for hotels and obtained the following results:
                Results:
                    {state["search_results"]}
                Your Task:
                You are at the final presentation phase of the process. You need to present the above results to the user in a very friendly manner.
            """),
            HumanMessage(content=state["search_results"])
        ])]
    }
    return {
        "messages": [state["search_results"]]
    }

In [13]:
from langgraph.graph import START, END
from langgraph.checkpoint.memory import MemorySaver

# The first argument is the unique node name
# The second argument is the function or object that will be called whenever
# the node is used.
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("extract_and_save", extract_and_save)
graph_builder.add_node("combine_requirements", combine_requirements)
graph_builder.add_node("process_results", process_results)


graph_builder.add_edge(START, "extract_and_save")
graph_builder.add_edge("extract_and_save", "combine_requirements")
graph_builder.add_edge("combine_requirements", "chatbot")
# graph_builder.add_conditional_edges(START, should_extract, {"extract": "extract_and_save", "chat": "chatbot"})
# The `tools_condition` function returns "tools" if the chatbot asks to use a tool, and "END" if
# it is fine directly responding. This conditional routing defines the main agent loop.
graph_builder.add_conditional_edges(
    "chatbot",
    route_tools,
    # The following dictionary lets you tell the graph to interpret the condition's outputs as a specific node
    # It defaults to the identity function, but if you
    # want to use a node named something else apart from "tools",
    # You can update the value of the dictionary to something else
    # e.g., "tools": "my_tools"
    {"tools": "tools", END: END, "extract": "extract_and_save", "chatbot": "chatbot"},
)
# Any time a tool is called, we return to the chatbot to decide the next step
graph_builder.add_edge("tools", "process_results")
graph_builder.add_edge("process_results", END)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [14]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [15]:
def extract_user_and_message(raw_message: str) -> tuple[str, str]:
    return [value.strip() for value in raw_message.split(":")]

async def handle_message_with_graph(raw_message, history):
    user, message = extract_user_and_message(raw_message)
    print(f"Extracted user: {user}")
    print(f"Extracted message: {message}")
    events = await graph.ainvoke({"messages": [{"role": "user", "content": raw_message}], "users": {
        "joseph": HotelSearchRequirements(),
        "saawan": HotelSearchRequirements()
    }, "input": message, "who": user, "extracted": False},
    config={"configurable": {"thread_id": 1}})
    print("This is the history: ")
    print(history)
    print("events: ")
    print(events)
    print("printing events: ")
    for event in events["messages"]:
        print("event: ")
        print(event)
    return events["messages"][-1].content

In [16]:
import gradio as gr

gr.ChatInterface(
    fn=handle_message_with_graph, 
    type="messages"
).launch()

/Users/joat/Documents/Projects/multiuserchat2/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


Extracted user: joseph
Extracted message: hello!
extract_and_Save called!
output: 
None
output_as_json: 
{}
setting extracted to true
updated extracted_data: 
{'users': {'joseph': HotelSearchRequirements(destination=None, num_adults=None, num_children=None, from_date=None, to_date=None, num_rooms=None, amenities=None, free_breakfast=None, free_cancelation=None, max_price=None, min_price=None), 'saawan': HotelSearchRequirements(destination=None, num_adults=None, num_children=None, from_date=None, to_date=None, num_rooms=None, amenities=None, free_breakfast=None, free_cancelation=None, max_price=None, min_price=None)}, 'extracted': True}
combine_requirements called!


Traceback (most recent call last):
  File "/Users/joat/Documents/Projects/multiuserchat2/env/lib/python3.12/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/joat/Documents/Projects/multiuserchat2/env/lib/python3.12/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/joat/Documents/Projects/multiuserchat2/env/lib/python3.12/site-packages/gradio/blocks.py", line 2137, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/joat/Documents/Projects/multiuserchat2/env/lib/python3.12/site-packages/gradio/blocks.py", line 1661, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/joat/Documents/Projects/multiuserchat2/en

In [17]:
while True:
    pass

KeyboardInterrupt: 

In [ ]:
config = {"configurable": {"thread_id": "1"}}

print(graph.get_state(config))


In [ ]:
for state in graph.get_state_history(config):
    print("Num Messages: ", len(state.values["messages"]), "Next: ", state.next)
    print("-" * 80)
    if len(state.values["messages"]) == 6:
        # We are somewhat arbitrarily selecting a specific state based on the number of chat messages in the state.
        to_replay = state

In [ ]:
import gradio as gr

messages = []

def sync_history():
    print("sync_history called!!")
    return messages

def track_messages(message, history):
    print("message: ")
    print(message)
    print("history: ")
    print(history)
    messages.append({
        "role": "user",
        "content": message
    })
    messages.append({
        "role": "assistant",
        "content": message
    })
    demo.chatbot_state.value = messages
    demo2.chatbot_state.value = messages
    demo.render()
    demo2.render()
    return message

scores = []

def track_score(score):
    scores.append(score)
    top_scores = sorted(scores, reverse=True)[:3]
    return top_scores

def store_message(message: str, history: list[str]):  
    output = {
        "Current messages": message,
        "Previous messages": history[::-1]
    }
    history.append(message)
    return output, history

state = gr.State(value=[])

# demo = gr.Interface(fn=store_message,
#                     inputs=["textbox", state],
#                     outputs=["json", gr.State()])

demo = gr.ChatInterface(
    track_messages,
    type="messages",
    chatbot=gr.Chatbot(value=sync_history, every=gr.Timer(1), type="messages")
)
demo.launch()

demo2 = gr.ChatInterface(
    track_messages,
    type="messages",
    chatbot=gr.Chatbot(value=sync_history, every=gr.Timer(1), type="messages")
)
demo2.launch()

In [ ]:
my_list = [1, 2, 3]

def changelist(li):
    li[:] = [4, 5, 6]

changelist(my_list)

print(my_list)

In [ ]:
import gradio as gr


def greet(name):
    return "Hello " + name + "!"


with gr.Blocks() as demo:
    output = gr.Chatbot()
    message = gr.Textbox(label="message")
    greet_btn = gr.Button("Greet")
    greet_btn.click(fn=greet, inputs=name, outputs=output, api_name="greet")

demo.launch()

In [ ]:
import gradio as gr
import random

def prefill_chatbot(choice):
    if choice == "Greeting":
        return [
            {"role": "user", "content": "Hi there!"},
            {"role": "assistant", "content": "Hello! How can I assist you today?"}
        ]
    elif choice == "Complaint":
        return [
            {"role": "user", "content": "I'm not happy with the service."},
            {"role": "assistant", "content": "I'm sorry to hear that. Can you please tell me more about the issue?"}
        ]
    else:
        return []

def random_response(message, history):
    return random.choice(["Yes", "No"])

with gr.Blocks() as demo:
    radio = gr.Radio(["Greeting", "Complaint", "Blank"])
    chat = gr.ChatInterface(random_response, type="messages")
    radio.change(prefill_chatbot, radio, chat.chatbot_value)

demo.launch()